In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [4]:
spark = SparkSession.builder \
    .appName("Pakistan_Ecom_Cleaning") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

26/04/17 11:32:35 WARN Utils: Your hostname, LAPTOP-KVJ76CML resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/17 11:32:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 11:32:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/17 11:32:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
file_path =r"/home/lst/my-spark/Pakistan Largest Ecommerce Dataset.csv"

In [6]:
df = spark.read.option("header", "true") \
               .option("inferSchema", "true") \
               .option("sep", ",") \
               .option("encoding", "utf-8") \
               .csv(file_path)

不手动打shcema

In [7]:
df = df.toDF(*[c.strip() for c in df.columns])
print(f"总行数: {df.count():,}")


总行数: 1,048,586


In [8]:
df.printSchema()
df.select("_c21").show(truncate=False)

root
 |-- item_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- price: double (nullable = true)
 |-- qty_ordered: string (nullable = true)
 |-- grand_total: string (nullable = true)
 |-- increment_id: string (nullable = true)
 |-- category_name_1: string (nullable = true)
 |-- sales_commission_code: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- Working Date: string (nullable = true)
 |-- BI Status: string (nullable = true)
 |-- MV: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Customer Since: string (nullable = true)
 |-- M-Y: string (nullable = true)
 |-- FY: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- _c21: string (nullable = true)
 |-- _c22: string (nullable = true)
 |-- _c23: string (nullable = true)
 |-- _c24: string (null

26/04/17 11:32:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: 
 Schema: _c21
Expected: _c21 but found: 
CSV file: file:///home/lst/my-spark/Pakistan%20Largest%20Ecommerce%20Dataset.csv


In [9]:
cols_to_drop = [col for col in df.columns if col.startswith("_c")]
cleaned_df = df.drop(*cols_to_drop)

In [10]:
cleaned_df.show(5,truncate=False)

+-------+--------------+----------+-----------------------------------------------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+
|item_id|status        |created_at|sku                                                        |price |qty_ordered|grand_total|increment_id|category_name_1  |sales_commission_code|discount_amount|payment_method|Working Date|BI Status|MV     |Year|Month|Customer Since|M-Y   |FY  |Customer ID|
+-------+--------------+----------+-----------------------------------------------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+
|211131 |complete      |7/1/2016  |kreations_YI 06-L                                          |1950.0|1          |1950      

In [11]:
cleaned_df.count()
print("列名:", cleaned_df.columns)

列名: ['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered', 'grand_total', 'increment_id', 'category_name_1', 'sales_commission_code', 'discount_amount', 'payment_method', 'Working Date', 'BI Status', 'MV', 'Year', 'Month', 'Customer Since', 'M-Y', 'FY', 'Customer ID']


In [12]:
from pyspark.sql.functions import sum as sp_sum

null_c= cleaned_df.select(
    [sp_sum(col(c).isNull().cast("int")).alias(c)
     for c in cleaned_df.columns]
)
null_c.show(5)

+-------+------+----------+------+------+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+------+------+------+--------------+------+------+-----------+
|item_id|status|created_at|   sku| price|qty_ordered|grand_total|increment_id|category_name_1|sales_commission_code|discount_amount|payment_method|Working Date|BI Status|    MV|  Year| Month|Customer Since|   M-Y|    FY|Customer ID|
+-------+------+----------+------+------+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+------+------+------+--------------+------+------+-----------+
| 464051|464066|    464051|464071|464062|     464062|     464062|      464062|         464226|               601233|         464062|        464062|      464062|   464062|464062|464062|464062|        464062|464073|464073|     464073|
+-------+------+----------+------+------+-----------+-----------+---

In [13]:
raw_text = spark.read.text(file_path)
raw_text.show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                      |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|item_id,status,created_at,sku,price,qty_ordered,grand_total,increment_id,category_name_1,sales_commission_code,discount_amount,payment_method,Working Date,BI Status, MV ,Year,Month,Customer Since,M-Y,FY,Customer ID,,,,,|
|211131,complete,7/1/2016,kreations_YI 06-L,1950,1,1950,100147443,Women's Fashion,\N,0,cod,7/1/2016,#REF!," 1,95

In [14]:
missing_df = cleaned_df.select([
    count(when(col(c).isNull(), c)).alias(f"{c}_null_count") for c in cleaned_df.columns
])

# 显示缺失值统计
print("=== 每列缺失值统计 ===")
missing_df.show(truncate=False)

=== 每列缺失值统计 ===
+------------------+-----------------+---------------------+--------------+----------------+----------------------+----------------------+-----------------------+--------------------------+--------------------------------+--------------------------+-------------------------+-----------------------+--------------------+-------------+---------------+----------------+-------------------------+--------------+-------------+----------------------+
|item_id_null_count|status_null_count|created_at_null_count|sku_null_count|price_null_count|qty_ordered_null_count|grand_total_null_count|increment_id_null_count|category_name_1_null_count|sales_commission_code_null_count|discount_amount_null_count|payment_method_null_count|Working Date_null_count|BI Status_null_count|MV_null_count|Year_null_count|Month_null_count|Customer Since_null_count|M-Y_null_count|FY_null_count|Customer ID_null_count|
+------------------+-----------------+---------------------+--------------+----------------+

In [15]:
cleaned_df =cleaned_df \
    .dropDuplicates() \
    .na.drop(subset=["created_at", "price", "qty_ordered", "grand_total", "item_id"])

missing_df = cleaned_df.select([
    count(when(col(c).isNull(), c)).alias(f"{c}_null_count") for c in cleaned_df.columns
])

# 显示缺失值统计
print("=== 每列缺失值统计 ===")
missing_df.show(truncate=False)


=== 每列缺失值统计 ===


+------------------+-----------------+---------------------+--------------+----------------+----------------------+----------------------+-----------------------+--------------------------+--------------------------------+--------------------------+-------------------------+-----------------------+--------------------+-------------+---------------+----------------+-------------------------+--------------+-------------+----------------------+
|item_id_null_count|status_null_count|created_at_null_count|sku_null_count|price_null_count|qty_ordered_null_count|grand_total_null_count|increment_id_null_count|category_name_1_null_count|sales_commission_code_null_count|discount_amount_null_count|payment_method_null_count|Working Date_null_count|BI Status_null_count|MV_null_count|Year_null_count|Month_null_count|Customer Since_null_count|M-Y_null_count|FY_null_count|Customer ID_null_count|
+------------------+-----------------+---------------------+--------------+----------------+----------------

In [16]:
pdf = missing_df.toPandas()
pdf
pdf = pdf.sort_values(by=0, axis=1, ascending=False)
pdf


,sales_commission_code_null_count,category_name_1_null_count,sku_null_count,status_null_count,Customer ID_null_count,M-Y_null_count,FY_null_count,item_id_null_count,created_at_null_count,price_null_count,...,grand_total_null_count,qty_ordered_null_count,Working Date_null_count,payment_method_null_count,discount_amount_null_count,BI Status_null_count,Month_null_count,Year_null_count,MV_null_count,Customer Since_null_count
0,137171,164,20,15,11,11,11,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:


cleaned_df=cleaned_df.na.fill({
        "category_name_1": "Other",
        "status": "unknow",
        "M-Y": "Unknown",
        "FY": "Unknown",
        "sku": "No_SKU",
        "sales_commission_code" :"no_code"
    })
cleaned_df =cleaned_df.na.drop(subset="Customer ID",)
cleaned_df.show(5,truncate=False)

+-------+--------+----------+-------------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+
|item_id|status  |created_at|sku                      |price |qty_ordered|grand_total|increment_id|category_name_1  |sales_commission_code|discount_amount|payment_method|Working Date|BI Status|MV     |Year|Month|Customer Since|M-Y   |FY  |Customer ID|
+-------+--------+----------+-------------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+
|211249 |received|7/1/2016  |kcc_glamour deal         |320.0 |1          |320        |100147522   |Beauty & Grooming|C-MUX-48271          |0              |cod           |7/1/2016    |Valid    | 320   |2016|7    |2016-7        |7-2016|FY17|43   

In [18]:
cleaned_df=cleaned_df.withColumn("created_at", to_timestamp(trim(col("created_at")), "M/d/yyyy")) \
    .withColumn("year", year("created_at")) \
    .withColumn("month", month("created_at")) \
    .withColumn("total_value", round(col("price") * col("qty_ordered"), 2))
cleaned_df.show(5,truncate=False)

+-------+--------+-------------------+-------------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+-----------+
|item_id|status  |created_at         |sku                      |price |qty_ordered|grand_total|increment_id|category_name_1  |sales_commission_code|discount_amount|payment_method|Working Date|BI Status|MV     |year|month|Customer Since|M-Y   |FY  |Customer ID|total_value|
+-------+--------+-------------------+-------------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+-----------+
|211249 |received|2016-07-01 00:00:00|kcc_glamour deal         |320.0 |1          |320        |100147522   |Beauty & Grooming|C-MUX-48271          |0              |cod           |7/

In [19]:
print("清洗后总行数:", cleaned_df.count())

清洗后总行数: 584513


In [20]:
missing_df = cleaned_df.select([
    count(when(col(c).isNull(), c)).alias(f"{c}") for c in cleaned_df.columns
])

# 显示缺失值统计
print("=== 每列缺失值统计 ===")
missing_df.show(truncate=False)

=== 每列缺失值统计 ===


+-------+------+----------+---+-----+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+---+----+-----+--------------+---+---+-----------+-----------+
|item_id|status|created_at|sku|price|qty_ordered|grand_total|increment_id|category_name_1|sales_commission_code|discount_amount|payment_method|Working Date|BI Status|MV |year|month|Customer Since|M-Y|FY |Customer ID|total_value|
+-------+------+----------+---+-----+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+---+----+-----+--------------+---+---+-----------+-----------+
|0      |0     |0         |0  |0    |0          |0          |0           |0              |0                    |0              |0             |0           |0        |0  |0   |0    |0             |0  |0  |0          |0          |
+-------+------+----------+---+-----+-----------+-----------+------------+----------

In [21]:
cleaned_df.printSchema()

root
 |-- item_id: string (nullable = true)
 |-- status: string (nullable = false)
 |-- created_at: timestamp (nullable = true)
 |-- sku: string (nullable = false)
 |-- price: double (nullable = true)
 |-- qty_ordered: string (nullable = true)
 |-- grand_total: string (nullable = true)
 |-- increment_id: string (nullable = true)
 |-- category_name_1: string (nullable = false)
 |-- sales_commission_code: string (nullable = false)
 |-- discount_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- Working Date: string (nullable = true)
 |-- BI Status: string (nullable = true)
 |-- MV: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- Customer Since: string (nullable = true)
 |-- M-Y: string (nullable = false)
 |-- FY: string (nullable = false)
 |-- Customer ID: string (nullable = true)
 |-- total_value: double (nullable = true)



In [22]:
output_path = "/home/lst/my-spark/cleaned"
cleaned_df.write.mode("overwrite").parquet(output_path)
print(f"数据已保存到：{output_path}")


数据已保存到：/home/lst/my-spark/cleaned


In [23]:

print("OK")

OK
